## Deep learning

In deze noteboek zullen we u rondleiden over hoe wij hebben gewerkt aan onze deep learning opdracht. 

## Imports

Voor ons project hebben we wat imports moeten doen. de in commentaar gezette imports was een transfer learning import voor `DenseNet121`. deze is niet doorgegaan maar wordt wel bijgehouden.

In [ ]:
%pip install matplotlib tensorflow keras mnist scikit-learn pandas opencv-python opencv-contrib-python

In [ ]:
# TensorFlow en tf.keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
# DenseNet alternatief, als je dit later will testen:
# from tensorflow.keras.applications import DenseNet121
# from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau
import cv2

# helper libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import pandas as pd
from PIL import Image
import os

## Data Voorbereiding & Verificatie

De dataset werd in het begin ingeladen vanuit `train.csv` en opgesplitst in een **trainingsset (80%)** en 
**validatieset (20%)** met behulp van stratificatie om een gelijke klassenverdeling te garanderen. Later wanneer starten met echt het model te trainenen schakkelde we over naar image_dataset_from_directory.

Beelden worden herschaald naar **384×384 pixels** en verwerkt in batches van **32**.
Om de trainingssnelheid te optimaliseren worden de datasets **gecached** (eenmalig ingeladen in geheugen)
en **geprefetcht** (volgende batch alvast klaargezet tijdens de huidige trainingsstap).

Omdat sommige leguanen sterk op elkaar lijken zoals de *Green anole* en *Brown anole* 
is het model gevoelig voor misclassificaties tussen visueel gelijkaardige soorten. 
Om dit te compenseren worden **class weights** toegepast, waardoor het model meer aandacht 
besteedt aan moeilijker te onderscheiden klassen en de algehele classificatienauwkeurigheid verbetert.


In [ ]:
SEED = 420
IMG_SIZE = 384
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE



# Oude manier van dataset ophalen test.csv
train_data = np.genfromtxt('train.csv', delimiter=',', skip_header=1, dtype='str')

train_x = train_data[:, 0]
train_y = train_data[:, 1].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    train_x,
    train_y,
    test_size=0.2,
    # random_state=SEED,
    stratify=train_y
)

# nieuwe manier van dataset ophalen image_dataset_from_directory
raw_train_dataset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

raw_validation_dataset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

class_names = raw_train_dataset.class_names
NUM_CLASSES = len(class_names)
print(class_names)

train_dataset = raw_train_dataset.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
validation_dataset = raw_validation_dataset.cache().prefetch(AUTOTUNE)

train_datatset = train_dataset
test_dataset = validation_dataset

# Class weights: hogere waarde = model besteedt meer aandacht aan die klasse
class_weight_dict = {
    0: 1.0,   # Black_spiny_tailed_iguana  → goed
    1: 1.5,   # Brown_anole                → matig
    2: 1.0,   # Cuban_knight_anole         → goed
    3: 1.5,   # Desert_iguana              → goed
    4: 1.5,   # Green_anole                → slecht
    5: 1.5,   # Green_iguana               → slecht
    6: 1.0,   # Lesser_Antillean_iguana    → goed
}

Uit de dataset komen de volgende classen naar boven allemaal met sparse labels (niet one hot encoded). Er zijn 7 classen in totaal met waarde 0-6.

<table>
<tr><th>Value</th><th>Class</th></tr>
<tr><td>0</td>	<td>Black_spiny_tailed_iguana</td></tr>
<tr><td>1</td>	<td>Brown_anole</td></tr>
<tr><td>2</td>	<td>Cuban_knight_anole</td></tr>
<tr><td>3</td>	<td>Desert_iguana</td></tr>
<tr><td>4</td>	<td>Green_anole</td></tr>
<tr><td>5</td>	<td>Green_iguana</td></tr>
<tr><td>6</td>	<td>Lesser_Antillean_iguana</td></tr>
</table>


## Transfer Learning met EfficientNetV2S

Als basismodel wordt **EfficientNetV2S** gebruikt, voorgetraind op ImageNet. 
De gewichten worden bevroren zodat enkel de nieuwe classificatielagen getraind worden.

De invoerbeelden worden eerst door een **data augmentatie**-stap geleid willekeurige spiegelingen, 
rotaties, zoom en contrastwijzigingen om overfitting te verminderen en de robuustheid van het model te verbeteren.

Bovenop het basismodel wordt een eenvoudig classificatiehoofd gebouwd:
een **GlobalAveragePooling2D**-laag reduceert de spatiale dimensies, gevolgd door twee **Dropout** en  **Dense**-lagen die uiteindelijk de 7 leguaansoorten classificeren via een softmax-activatie.

> **DenseNet121** werd ook getest. Hoewel DenseNet in theorie beter geschikt is voor kleinere datasets 
> door zijn feature-hergebruik, behaalde het in de praktijk lagere scores dan EfficientNetV2S op deze dataset.
> Daarom werd uiteindelijk gekozen voor EfficientNetV2S.

In [ ]:
# Transfer learning met EfficientNetV2S.
# Dit volgt het transfer learning voorbeeld: frozen pretrained body + GlobalAveragePooling2D + simple Dense head.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.2),
], name="data_augmentation")

base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    layers.Lambda(preprocess_input, name="efficientnetv2_preprocess"), # Roep preproccessing op voor efficientnet

    # Test met kleuren kannaal
    # layers.DepthwiseConv2D((3,3), depth_multiplier=1,activation='relu', padding="same"),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

## Compilatie & F1-Score Metric

Het model wordt gecompileerd met **Adam** als optimizer en **sparse categorical crossentropy** als verliesfunctie.

Als evaluatiemetric wordt de **macro F1-score** gebruikt in plaats van enkel accuracy, 
omdat dit een eerlijker beeld geeft over alle klassen ook de moeilijkere.
Omdat Keras dit niet standaard ondersteunt voor integer labels, werd een kleine 
aangepaste metric `SparseF1` geschreven.

In [ ]:
# Compileer model met EfficientNetV2S frozen body .
# We trainen nu enkel de head van het model wat stabieler is voor nu voor deze kleine dataset.
from tensorflow.keras.metrics import F1Score

class SparseF1(tf.keras.metrics.Metric):
    def __init__(self, name='f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.f1 = F1Score(average='macro')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
        self.f1.update_state(y_true, y_pred, sample_weight)

    def result(self):
        return self.f1.result()

    def reset_states(self):
        self.f1.reset_states()

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=[SparseF1(), "accuracy"]
)

## Training

Het model wordt getraind over **12 epochs** met twee callbacks:

**ReduceLROnPlateau** verlaagt automatisch de learning rate wanneer de val F1-score 
niet meer verbetert, zodat het model fijner kan bijsturen.

**RestoreBestValidationWeights** houdt de beste gewichten bij tijdens het trainen 
en herstelt deze aan het einde zo eindigen we altijd met het best presterende model, 
ook als de laatste epoch slechter was. 

In [ ]:
# Train voor exact 12 epochs.
# Validatie scores kunnen snel veranderen dus deze callback functie herstelt de beste weights na het hele model te trainen.
class RestoreBestValidationWeights(Callback):
    def __init__(self, monitor='val_accuracy'):

        # Zet variables om te monitoren
        super().__init__()
        self.monitor = monitor
        self.best_value = -np.inf
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        current_value = logs.get(self.monitor)
        # Als er nieuwe hogere waarde voor gemonitorede metric er is onthoud deze
        if current_value is not None and current_value > self.best_value:
            self.best_value = current_value
            self.best_weights = self.model.get_weights()
            self.best_epoch = epoch + 1

    def on_train_end(self, logs=None):
        # Reset gewichten naar hoogste waarde
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)
            print(f"Restored epoch {self.best_epoch} weights with best {self.monitor}: {self.best_value:.4f}")

callbacks = [
    # Verlaagt de learning rate automatisch als val_f1 niet meer verbetert.
    # Dit helpt het model om fijner te leren wanneer het vast lijkt te zitten.
    ReduceLROnPlateau(
        monitor='val_f1',      # kijkt naar de validation F1-score
        factor=0.3,            # elke keer dat het triggert: learning rate × 0.3 (dus 70% kleiner)
        patience=2,            # wacht 2 epochs zonder verbetering voordat het ingrijpt
        min_lr=1e-7,           # de learning rate mag niet kleiner worden dan 0.0000001
        mode="max"
    ),
    RestoreBestValidationWeights(monitor='val_f1')
]

history = model.fit(
    train_dataset,
    class_weight=class_weight_dict,
    validation_data=validation_dataset,
    epochs=12,
    batch_size=10,
    callbacks=callbacks
)

## Trainingscurves

De loss en accuracycurves worden gevisualiseerd over alle epochs. 
Dit geeft een beeld van hoe het model evolueert tijdens het trainen 
en of er sprake is van **overfitting** (trainingsaccuracy stijgt, validatieaccuracy daalt).

In [ ]:
def plotLosses(history):
  # Create a figure and a grid of subplots with a single call
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5))
  # Plot de loss curves op de eerste subplot
  ax1.plot(history.history['loss'], label='training loss')
  ax1.plot(history.history['val_loss'], label='validation loss')
  ax1.set_title('Loss curves')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Loss')
  ax1.legend()
  # Plot de accuracy curves op de tweede subplot
  ax2.plot(history.history['accuracy'], label='training accuracy')
  ax2.plot(history.history['val_accuracy'], label='validation accuracy')
  ax2.set_title('Accuracy curves')
  ax2.set_xlabel('Epoch')

  ax2.set_ylabel('Accuracy')
  ax2.legend()
  # Pas spacing tussen subplots aan
  fig.tight_layout()
  # Toon de figure
  plt.show()


plotLosses(history)

## Confusion matrix

Om te kijken waar hij exact verkeerd gokt. hier hebben we dan de verschillende aanpassingen aan gemaakt zoals `Class_weight` in het begin van de code. de meeste voorkomende fouten worden vanonder ook nog gegeven. 

<table>
<tr><th>Value</th><th>Class</th></tr>
<tr><td>0</td>	<td>Black_spiny_tailed_iguana</td></tr>
<tr><td>1</td>	<td>Brown_anole</td></tr>
<tr><td>2</td>	<td>Cuban_knight_anole</td></tr>
<tr><td>3</td>	<td>Desert_iguana</td></tr>
<tr><td>4</td>	<td>Green_anole</td></tr>
<tr><td>5</td>	<td>Green_iguana</td></tr>
<tr><td>6</td>	<td>Lesser_Antillean_iguana</td></tr>
</table>


In [ ]:
# Confusion matrix: Toon aan welke klasse het model met elkaar verwart.
y_true = []
y_pred = []

for images, labels in validation_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

# Gebruik nummers (index nummers) 0-6 in the plot, voor leesbaarheid.
# De tabel hierboven is een herinerring aan de index labels per classe.
label_numbers = np.arange(NUM_CLASSES)

plt.figure(figsize=(8, 8))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_numbers)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix")
plt.show()

# Extra overzicht: print de meest gemaakte fouten.
mistakes = []
for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        if true_class != predicted_class and cm[true_class, predicted_class] > 0:
            mistakes.append((cm[true_class, predicted_class], true_class, predicted_class))

mistakes = sorted(mistakes, reverse=True)
for count, true_label, predicted_label in mistakes[:10]:
    print(f"{count}x: label {true_label} predicted as label {predicted_label}")

## Voorspellingen

Het getrainde model genereert voorspellingen op de validatieset. 
Per afbeelding wordt een kans teruggegeven voor elke klasse de klasse met de hoogste kans is de uiteindelijke voorspelling.

In [ ]:
predictions = model.predict(validation_dataset)
print(np.argmax(predictions, axis=1))

## Foutieve Voorspellingen d.m.v. Grad-CAM Visualisatie

Om beter te begrijpen waar het model naar kijkt bij een voorspelling, wordt **Grad-CAM** gebruikt.
Dit kleurt de zones in de afbeelding die het meest bijdroegen aan de beslissing, 
warme kleuren (rood/geel) betekenen veel aandacht, koele kleuren (blauw) weinig.

Dit wordt enkel toegepast op de **10 meest zelfzekere foute voorspellingen**, 
zodat we kunnen zien waarom het model de mist in ging.

In [ ]:
# Start lijst met afbeelding pixel waarde (X_test) en werkalijke labels (y_test)
X_test = []
y_test = []

# Validatie data bestaat uit twee waarde: de batch afbeeldingen en labels. Deze worden naar een array om gezet.
for batch_images, batch_labels in validation_dataset:
    X_test.append(batch_images.numpy())
    y_test.append(batch_labels.numpy())
X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

# Maak voorspellingen en haal de hoogste waarde op (voorspelde waarde)
predictions = model.predict(tf.data.Dataset.from_tensor_slices(X_test).batch(BATCH_SIZE), verbose=0)
pred = np.argmax(predictions, axis=1)

# Haal enkel de fouten voorspellingen en bepaal lijst met tuples met info van deze vorrspelling + reverse het voor meest foute eerste hebben
wrong_indices = np.where(pred != y_test)[0]
wrong_predictions = []
for wrong_index in wrong_indices:
    prediction = np.argmax(predictions[wrong_index])
    test_label = y_test[wrong_index]
    prediction_prob = np.max(predictions[wrong_index])
    wrong_predictions.append((wrong_index, prediction, test_label, prediction_prob))
wrong_predictions.sort(key=lambda x: x[3], reverse=True)

# Grad-CAM helpfunctie
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="top_activation"):

    # Neem basis model om de geslecteerde afbeeldingen door te runnen wat hier gebeuren de convoluties.
    base_model = model.get_layer("efficientnetv2-s")

    # bepaal grad model (model om te vinden waar ons model op focussed) met het basis model
    # Het heeft de laaste convolutie laag nodig als deel van zijn output parameter
    grad_model = tf.keras.models.Model( inputs=base_model.inputs, outputs=[
         base_model.get_layer(last_conv_layer_name).output,
           base_model.output ] )

    img_preprocessed = preprocess_input(tf.cast(img_array, tf.float32))

    # GradientTape wordt gebruikt om alle berekeningen tijdelijk op te slaan,
    # zodat TensorFlow later de gradients kan berekenen.
    # Dit is nodig voor technieken zoals Grad-CAM, waar we willen zien
    # welke delen van de input het meest invloed hebben op de voorspelling.
    # Alles binnen dit blok wordt "getracked" voor automatische differentiatie.
    with tf.GradientTape() as tape:
        # Maak voorspellingen op alle meegegeven afbeeldingen
        conv_outputs, base_out = grad_model(img_preprocessed)

        # Stuur door de rest van het Sequential model (na base_model)
        x = base_out
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D):
                x = layer(x)
            elif isinstance(layer, tf.keras.layers.Dropout):
                x = layer(x, training=False)
            elif isinstance(layer, tf.keras.layers.Dense):
                x = layer(x)

        # Haal meest zekere waarde op + gebruik het voor de afbeeldingen te verkrijgen
        pred_index = tf.argmax(x[0])
        class_channel = x[:, pred_index]

    # Bepaald welke feature het belangrijkste zijn -> hoe elke pixel het eind resultaat beinvloed. Haalt de waarde op via tape.
    grads = tape.gradient(class_channel, conv_outputs)

    # Bepaal gemiddeld hoe belangrijk elke feature map is. Hierdoor behouden we de belangrijkste.
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Via gebruik van matrix vermenigvuldiging "@" worden de convolutie lagen met hun gewichten vermenigvuldigt -> Hierdoor blijft nog 1 feature map over (feature map met de belangrijkste features)
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]

    # Squeeze om de 3d tensor naar 2D array om te zetten (want enkel 1 feature map)
    heatmap = tf.squeeze(heatmap)

    # Behoud enkel de positieve waarde + normalizeer de data
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Vervang met de naam die uit de printloop hierboven komt, waarschijnlijk:
# last_conv_layer_name = "top_activation"  # of "top_conv", "block6o_add", ...


# How many wrong predictions?
top_wrong = 10
fig, axes = plt.subplots(top_wrong, 3, figsize=(18, top_wrong * 5))

col_titles = ["Originele afbeelding", "Grad-CAM heatmap", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=14, fontweight='bold')

for i in range(top_wrong):
    idx, pred_class, true_class, prob = wrong_predictions[i]
    img_array = X_test[idx]
    img_expanded = np.expand_dims(img_array, axis=0)

    # Grad-CAM
    heatmap = make_gradcam_heatmap(img_expanded, model)

    # Herschaal heatmap naar afbeelding grote
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))

    # Bepaal kleuren voor als overlay te gebruiken
    heatmap_colored = np.uint8(255 * plt.cm.jet(heatmap_resized)[:, :, :3])
    original = np.uint8(img_array)
    superimposed = cv2.addWeighted(original, 0.6, heatmap_colored, 0.4, 0)

    xlabel_text = (
        f"Actual: {class_names[true_class]}  →  "
        f"Predicted: {class_names[pred_class]}  "
        f"(prob: {prob:.2%})"
    )

    # Kolom 1: origineel
    axes[i, 0].imshow(original)
    axes[i, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # Kolom 2: heatmap
    axes[i, 1].imshow(heatmap_resized, cmap='jet')
    axes[i, 1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    axes[i, 1].set_xlabel(xlabel_text, color='red', fontsize=11)

    # Kolom 3: overlay
    axes[i, 2].imshow(superimposed)
    axes[i, 2].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

## Fine tunning

Met deze informatie van wat er fout kan gaan word het model nog eens opnieuw getraind. Dit wordt gedaan wia fine tunning. Hierbij ontdooien we een deel van ons basis model om deze dan met een kleine learning rate aan te passen voor een nieuwe taak.

In [ ]:
base_model.trainable = True

# unfreeze top most layers
for layer in base_model.layers[:-100]:
    layer.trainable = False

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=1e-5),
    metrics=[SparseF1(), "accuracy"]
)
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=12,
    batch_size=10,
    callbacks=callbacks
)


In [ ]:
plotLosses(history)

## Beter resultaat

Door deze veranderingen kun je zien dat er minder verkeerd word geraden door de AI. Wat een goed teken is.

In [ ]:
# Confusion matrix: shows which classes the model confuses with each other.
y_true = []
y_pred = []

for images, labels in validation_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

# Use numbers 0-6 in the plot, so the matrix stays readable.
# The table above shows which number belongs to which class.
label_numbers = np.arange(NUM_CLASSES)

plt.figure(figsize=(8, 8))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_numbers)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix")
plt.show()

# Extra overview: print the most common mistakes.
mistakes = []
for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        if true_class != predicted_class and cm[true_class, predicted_class] > 0:
            mistakes.append((cm[true_class, predicted_class], true_class, predicted_class))

mistakes = sorted(mistakes, reverse=True)
for count, true_label, predicted_label in mistakes[:10]:
    print(f"{count}x: label {true_label} predicted as label {predicted_label}")

## Voorspellingen op Testset

De testafbeeldingen worden ingeladen vanuit `test.csv`, herschaald naar **384×384 pixels** 
en in dezelfde volgorde verwerkt als in het CSV-bestand.

Het model genereerd vervolgens een voorspelling per afbeelding
de klasse met de hoogste kans wordt als eindvoorspelling genomen.

In [ ]:
# Genereer voorspellingen voor de echte test afbeeldingen in de zelfde volgorde als de test.csv.
test_ids = pd.read_csv("test.csv")["id"].astype(str).tolist()
test_paths = [os.path.join("test", f"{image_id}.jpg") for image_id in test_ids]

def load_test_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return tf.cast(image, tf.float32)

test_prediction_dataset = (
    tf.data.Dataset.from_tensor_slices(test_paths)
    .map(load_test_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

predictions = model.predict(test_prediction_dataset)
pred = np.argmax(predictions, axis=1)

## Submission file aanmaken
Submission file word hier aangemaakt. Dit word in een csv formaat gedaan, in het verwachten formaat voor Kaggle.

In [ ]:
# Match het formaat van sample_submission.csv: id,label
df = pd.DataFrame(data={"id": test_ids, "label": pred.flatten()})
# df.to_csv("csv/test_submission.csv", index=False)
df.head()

## Verwijzingen

Een deel van het testen werd uitgevoerd in Kaggle en Google Colab. Dit maakte het eenvoudiger om de epochs uit te voeren, aangezien dit lokaal veel meer tijd kon kosten.

Voor dit project hebben we met ons drie verschillende testen uitgevoerd om de F1-score en accuracy te verbeteren. Eerst werkten we met de file [higher accuracy](Extra/Extra1.ipynb). In deze file hebben we verschillende transfer learning-modellen getest en vergeleken. Uiteindelijk kwamen we uit bij EfficientNetV2S als beste keuze.

Daarnaast gebruikten we de [start file](Extra/Extra2.ipynb). Dit was de basisfile met code uit de cursus, die diende als startpunt voor het project.

## Extra uitproberingen

### Model keuze

Zoals eerder vermeld werd, werd er in het begin gekozen tussen welk model te gebruiken. Hierbij werd er vooral gekeken tussen **DenseNet121** en **EfficientNetV2S**. Zoals daarin ook vermeldt wert **EfficientNetV2S** gekozen voor zijn hogere resultaten.

### Kleuren kannaal.

Een optie die tijden de les was geven om uit te proberen was om het kleuren kannaal te behouden. Normaal gezien heeft dit weinig zien en kost veel meer reken kracht, maar voor deze dataset zou het mogelijk nuttig zijn. Dit komt doordat sommige leguanen goed camoufleren in de achtergrond waardoor mogelijk via het kleuren kanaal een betere clasificatie gemaakt kon worden. Na het inspecteren via Grad-cam en het uitproberen via layers.DepthwiseConv2D() konden we zien dat het model in de meeste gevallen de leguaan goed kon vinden en dat kleur toevoegen het model vaak een slechteren f1-score gaf.

## AI

Voor deep learning is er wat AI gebruikt. AI werd het meeste gebruikt voor verschillende transfer learning methodes te testen en uit te laten leggen wat bepaalde stukken code deed. 

Een voorbeeld van hoe we AI gebruikte was bijvoorbeeld hoe we tot het idee kwammen om nog te fine tunnen. In de onderstaande prompt kan je een deel van het idee zien.

<img src="AI_policy_images\Prompt.png" alt="Description" width="500">
<img src="AI_policy_images\output.png" alt="Description" width="500">